# ACSS — Protocol Parsing & Fact-Based Validation

Live demo, two parts:
1. **Parse** a real synthesis protocol's raw text into a hierarchical task DAG (material-flow dependencies, not text order) — the LLM reads the prose directly, nothing is pre-extracted for it.
2. **Validate** a reagent's hidden role: an LLM proposes it, a separate judge checks it against real retrieved evidence before it's trusted.


## Part 1 — Protocol Text → DAG


In [ ]:
import os, json, textwrap
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown

from pipeline import run_round1_from_text, run_round2, EQUIPMENT_COLORS, CATEGORY_COLORS

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise EnvironmentError("ANTHROPIC_API_KEY not set -- see .env.template")
print("[ok] Anthropic key found")

with open("demo_protocols.json") as f:
    DEMOS = json.load(f)
demo = DEMOS[0]
print(f"[ok] loaded demo protocol: {demo['label']}")


In [ ]:
print(f"Target  : {demo['target']}")
print(f"Type    : {demo['synthesis_type']}")
print(f"Source  : {demo['source']}")
print()
print("Protocol text (this is the ENTIRE input -- no pre-extracted operations,")
print("no conditions table, no precursor list; the LLM reads this prose directly):")
print("-" * 78)
print(demo["protocol_text"])
print("-" * 78)


### Round 1 -- LLM reads the prose above and parses it into a Task DAG (material-flow dependencies, not textual order)


In [ ]:
r1_result = run_round1_from_text(
    demo["protocol_text"],
    target=demo["target"],
    synthesis_type=demo["synthesis_type"],
    source=demo["source"],
    verbose=True,
)
print("\n--- LLM reasoning ---")
print(textwrap.fill(r1_result["reasoning"], width=90))


In [ ]:
def draw_task_dag(r1, title="Task DAG"):
    G = nx.DiGraph()
    nodes, edges = r1["nodes"], r1["edges"]
    for n in nodes:
        G.add_node(n["id"], **n)
    for e in edges:
        G.add_edge(e["from"], e["to"], material=e["material"])

    try:
        layers = list(nx.topological_generations(G))
    except nx.NetworkXUnfeasible:
        layers = [[n] for n in G.nodes()]

    pos = {}
    for li, layer in enumerate(layers):
        for ni, node in enumerate(sorted(layer)):
            pos[node] = (li * 2.5, -(ni - (len(layer) - 1) / 2) * 1.8)

    node_colors = []
    for n in G.nodes():
        op_types = G.nodes[n].get("operation_types", [])
        node_colors.append(CATEGORY_COLORS.get(op_types[0] if op_types else None, "#BDC3C7"))

    fig, ax = plt.subplots(figsize=(max(12, len(layers) * 2.8), 7))
    ax.set_title(f"{title}  --  {r1['target_material']} [{r1['synthesis_type']}]", fontsize=14, fontweight="bold", pad=15)

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2400, alpha=0.92, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={n: n for n in G.nodes()}, font_size=11, font_weight="bold", ax=ax)

    label_pos = {k: (v[0], v[1] - 0.55) for k, v in pos.items()}
    nx.draw_networkx_labels(G, label_pos,
        labels={n["id"]: "\n".join(textwrap.wrap(n["label"], 18)) for n in nodes}, font_size=7.5, ax=ax)

    nx.draw_networkx_edges(G, pos, edge_color="#555", arrows=True, arrowsize=20, width=1.8,
                           connectionstyle="arc3,rad=0.08", ax=ax)
    edge_labels = {(e["from"], e["to"]): e["material"] for e in edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, label_pos=0.35, ax=ax)

    patches = [mpatches.Patch(color=c, label=k.replace("Operation", "")) for k, c in CATEGORY_COLORS.items()]
    ax.legend(handles=patches, loc="upper right", fontsize=8, title="Operation type", title_fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


draw_task_dag(r1_result, title="Round 1 -- Task DAG")


### Round 2 -- each task decomposed into atomic, equipment-tagged operations

In [ ]:
r2_result = run_round2(r1_result, verbose=True)


In [ ]:
def draw_hierarchical_dag(r2):
    nodes, edges = r2["nodes"], r2["edges"]

    G_task = nx.DiGraph()
    for n in nodes:
        G_task.add_node(n["id"], **{k: v for k, v in n.items() if k != "atomic_ops"})
    for e in edges:
        G_task.add_edge(e["from"], e["to"])

    try:
        layers = list(nx.topological_generations(G_task))
    except nx.NetworkXUnfeasible:
        layers = [[n] for n in G_task.nodes()]

    task_pos = {}
    for li, layer in enumerate(layers):
        for ni, node in enumerate(sorted(layer)):
            task_pos[node] = (li * 2.5, -(ni - (len(layer) - 1) / 2) * 2.0)

    task_colors = [CATEGORY_COLORS.get((G_task.nodes[n].get("operation_types") or [None])[0], "#BDC3C7")
                   for n in G_task.nodes()]

    expand_node = max(nodes, key=lambda n: len(n.get("atomic_ops", [])))

    G_atom = nx.DiGraph()
    for aop in expand_node["atomic_ops"]:
        G_atom.add_node(aop["id"], **aop)
    for aop in expand_node["atomic_ops"]:
        for dep in aop.get("depends_on", []):
            if dep in G_atom:
                G_atom.add_edge(dep, aop["id"])

    try:
        atom_layers = list(nx.topological_generations(G_atom))
    except nx.NetworkXUnfeasible:
        atom_layers = [[n] for n in G_atom.nodes()]

    atom_pos = {}
    for li, layer in enumerate(atom_layers):
        for ni, node in enumerate(sorted(layer)):
            atom_pos[node] = (li * 2.2, -(ni - (len(layer) - 1) / 2) * 1.6)

    atom_colors = [EQUIPMENT_COLORS.get(G_atom.nodes[n].get("equipment"), "#ECF0F1") for n in G_atom.nodes()]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.2, 1.6]})
    fig.suptitle(f"Hierarchical DAG -- {r2['target_material']} [{r2['synthesis_type']}]", fontsize=14, fontweight="bold")

    ax1.set_title("Round 1: Task DAG\n(boxed node expanded ->)", fontsize=11, pad=8)
    nx.draw_networkx_nodes(G_task, task_pos, node_color=task_colors, node_size=1800, alpha=0.9, ax=ax1)
    nx.draw_networkx_labels(G_task, task_pos, labels={n: n for n in G_task.nodes()}, font_size=10, font_weight="bold", ax=ax1)
    lpos = {k: (v[0], v[1] - 0.5) for k, v in task_pos.items()}
    nx.draw_networkx_labels(G_task, lpos,
        labels={n["id"]: "\n".join(textwrap.wrap(n["label"], 15)) for n in nodes}, font_size=7, ax=ax1)
    nx.draw_networkx_edges(G_task, task_pos, edge_color="#444", arrows=True, arrowsize=18, width=1.6,
                           connectionstyle="arc3,rad=0.08", ax=ax1)
    nx.draw_networkx_nodes(G_task, task_pos, nodelist=[expand_node["id"]], node_color="none",
                           node_size=2200, edgecolors="black", linewidths=3, ax=ax1)
    cat_patches = [mpatches.Patch(color=c, label=k.replace("Operation", "")) for k, c in CATEGORY_COLORS.items()]
    ax1.legend(handles=cat_patches, loc="lower left", fontsize=7, title="Op type", title_fontsize=8)
    ax1.axis("off")

    ax2.set_title(f"Round 2: Atomic ops for {expand_node['id']} -- '{expand_node['label']}'", fontsize=11, pad=8)
    nx.draw_networkx_nodes(G_atom, atom_pos, node_color=atom_colors, node_size=1600, alpha=0.92, ax=ax2)
    atom_labels = {aop["id"]: f"{aop['xdl_action']}\n{aop['duration_estimate']['value']}{aop['duration_estimate']['unit']}"
                   for aop in expand_node["atomic_ops"]}
    nx.draw_networkx_labels(G_atom, atom_pos, labels=atom_labels, font_size=8, ax=ax2)
    alpos = {k: (v[0], v[1] - 0.55) for k, v in atom_pos.items()}
    short_desc = {aop["id"]: "\n".join(textwrap.wrap(aop["label"], 16)) for aop in expand_node["atomic_ops"]}
    nx.draw_networkx_labels(G_atom, alpos, labels=short_desc, font_size=6.5, ax=ax2)
    nx.draw_networkx_edges(G_atom, atom_pos, edge_color="#444", arrows=True, arrowsize=16, width=1.4,
                           connectionstyle="arc3,rad=0.08", ax=ax2)
    used_equip = {aop["equipment"] for aop in expand_node["atomic_ops"]}
    equip_patches = [mpatches.Patch(color=EQUIPMENT_COLORS.get(eq, "#ECF0F1"), label=(eq or "none").replace("_", " "))
                     for eq in sorted(used_equip, key=lambda x: x or "")]
    ax2.legend(handles=equip_patches, loc="lower left", fontsize=7, title="Equipment", title_fontsize=8)
    ax2.axis("off")

    plt.tight_layout()
    plt.show()


draw_hierarchical_dag(r2_result)


In [ ]:
from collections import Counter

equip_count = Counter()
for node in r2_result["nodes"]:
    for aop in node.get("atomic_ops", []):
        if aop.get("equipment"):
            equip_count[aop["equipment"]] += 1

targets = {e["to"] for e in r2_result["edges"]}
root_nodes = [n for n in r2_result["nodes"] if n["id"] not in targets]

print(f"Task nodes      : {len(r2_result['nodes'])}")
print(f"Dependency edges: {len(r2_result['edges'])}")
total_ops = sum(len(n.get('atomic_ops', [])) for n in r2_result['nodes'])
print(f"Total atomic ops: {total_ops}")
if len(root_nodes) > 1:
    print(f"\n{len(root_nodes)} PARALLEL entry tasks (no dependencies): {[r['id'] for r in root_nodes]}")

print("\nEquipment usage across all atomic ops:")
for eq, count in equip_count.most_common():
    print(f"  {eq:<35} {'#' * count} ({count})")


## Part 2 — Fact-Based Validation

For a reagent, does an LLM's guessed hidden role survive an independent check against real retrieved evidence? An unverified claim never becomes a fact in this pipeline.

In [ ]:
import sys
from pathlib import Path
import anthropic, openai
from llm_cache import call_with_cache, parse_json_response

REFERENCE_PATH = Path("validation_reference.json")
EVIDENCE_PATH = Path("validation_evidence.json")

claude_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"), timeout=45.0, max_retries=0)
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=45.0, max_retries=0)
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

GENERATE_SYSTEM = """You are a domain expert in inorganic materials chemistry (hydrothermal, \
co-precipitation, sol-gel synthesis). Given a precursor material, decide whether it \
plausibly plays a role BEYOND simply being the source of its constituent elements -- \
e.g. a pH-modifying agent, an oxidizing/reducing agent, a capping/structure-directing \
agent, a slow-release hydrolysis agent, etc.

Do NOT force a finding. Most simple ionic precursors (e.g. a plain metal chloride) \
really are just the source of that metal -- for those, say so plainly. Only propose a \
hypothesis when there is a genuine, real mechanistic role you have real chemistry \
knowledge of, not a speculative guess.

Respond with ONLY a JSON object, no other text:
{"has_hypothesis": true|false, "role_label": "<short tag, or null>", "mechanism_description": "<1-2 sentences, or null>", "search_query": "<a good web search query to find literature confirming this, or null>"}"""

JUDGE_SYSTEM = """You are verifying whether retrieved evidence genuinely supports a specific chemistry claim about a precursor material's role in synthesis.

Respond with ONLY a JSON object, no other text:
{"supported": true|false, "confidence": "high"|"medium"|"low", "reasoning": "one to two sentences, citing what the evidence does or does not confirm"}

Be precise: evidence that confirms the material is USED in this kind of synthesis is not the same as evidence that confirms the SPECIFIC MECHANISTIC ROLE claimed. If the evidence only shows general usage without confirming the specific claimed mechanism, mark supported=false or confidence=low, and say so in your reasoning."""


def generate_hypothesis(formula, name, cache_key):
    user = (f"Precursor formula: {formula}\nCommon/IUPAC name: {name}\n"
            f"Context: solution-based inorganic materials synthesis (hydrothermal or precipitation).")

    def live():
        response = claude_client.messages.create(
            model=CLAUDE_MODEL, max_tokens=1024, system=GENERATE_SYSTEM,
            messages=[{"role": "user", "content": user}])
        text_block = next(b for b in response.content if b.type == "text")
        return parse_json_response(text_block.text)

    return call_with_cache(cache_key, live)


def judge_hypothesis(formula, role, mechanism, evidence_text, cache_key):
    user = f"Claim: {formula} ({role}) -- {mechanism}\n\nRetrieved evidence: {evidence_text}"

    def live():
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL, max_completion_tokens=1024,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}])
        return parse_json_response(response.choices[0].message.content)

    return call_with_cache(cache_key, live)


with open(REFERENCE_PATH) as f:
    REFERENCE = json.load(f)
with open(EVIDENCE_PATH) as f:
    EVIDENCE = json.load(f)

print(f"[ok] {len(REFERENCE)} historical reference records, {len(EVIDENCE)} evidence records loaded")

HIST_BADGE = {True: "PASS", False: "FAIL", None: "declined"}
LIVE_BADGE = {True: "PASS", False: "FAIL", None: "DECLINED"}


def run_case_live(material_id, name):
    hist = REFERENCE[str(material_id)]
    formula = hist["formula"]

    lines = [f"### `{formula}`", "",
             f"**Historical run:** " + HIST_BADGE[hist["historical_pass"] if hist["has_hypothesis"] else None]
             + (f" -- role `{hist['role_label']}`" if hist["has_hypothesis"] else ""),
             "", "---", ""]

    gen = generate_hypothesis(formula, name, f"acss_{material_id}_generate")
    lines.append(f"**Generate (live Claude, `{gen.get('_source')}`):**")
    if not gen["has_hypothesis"]:
        lines.append("declined -- no hidden role proposed.")
        lines.append(f"\n**Live outcome:** {LIVE_BADGE[None]}")
        display(Markdown("\n\n".join(lines)))
        return

    lines.append(f"proposed role `{gen['role_label']}`")
    lines.append(f"> {gen['mechanism_description']}")

    ev = EVIDENCE.get(str(material_id))
    if ev is None:
        lines.append("\n**Ground:** no independently retrieved evidence exists for this material "
                      "-- stays **unverified**, per the rule that an unverified claim never becomes a fact.")
        lines.append(f"\n**Live outcome:** UNVERIFIED (no evidence to judge against)")
        display(Markdown("\n\n".join(lines)))
        return

    lines.append(f"\n**Ground (real web search, no LLM):** query -- *{ev['query']}*")
    lines.append(f"> {ev['evidence']}")
    lines.append(f">\n> Source: {ev['sources'][0]}")

    judge = judge_hypothesis(formula, gen["role_label"], gen["mechanism_description"], ev["evidence"],
                              f"acss_{material_id}_judge")
    lines.append(f"\n**Judge (live GPT, `{judge.get('_source')}`):** confidence *{judge['confidence']}*")
    lines.append(f"> {judge['reasoning']}")
    lines.append(f"\n**Live outcome:** {LIVE_BADGE[judge['supported']]}")

    display(Markdown("\n\n".join(lines)))


In [ ]:
run_case_live(2, "titanium tetrachloride")   # historical PASS


In [ ]:
run_case_live(5, "iron(III) chloride")       # historical FAIL


In [ ]:
run_case_live(13, "silver nitrate")          # historical DECLINED


In [ ]:
total = len(REFERENCE)
generated = sum(1 for v in REFERENCE.values() if v["has_hypothesis"])
passed = sum(1 for v in REFERENCE.values() if v["historical_pass"])

display(Markdown(
    f"**{total} materials in -> {generated} hypotheses generated -> {passed} survived verification "
    f"({(1 - passed / generated):.0%} rejection rate).** A judge that agreed with everything the "
    f"generator proposed would carry zero information -- what gets rejected is the finding."
))


### Optional, time permitting — fully live on an unseen material

In [ ]:
live_material = "Ba(NO3)2"
live_name = "barium nitrate"
live_evidence = None  # set to real retrieved text to judge it; None shows the unverified path

gen = generate_hypothesis(live_material, live_name, "acss_bonus_generate")
lines = [f"### `{live_material}` -- fully live", "", f"**Generate (live Claude, `{gen.get('_source')}`):**"]

if not gen["has_hypothesis"]:
    lines.append("declined -- no hidden role proposed.")
elif live_evidence is None:
    lines.append(f"proposed role `{gen['role_label']}`")
    lines.append(f"> {gen['mechanism_description']}")
    lines.append("\n**Ground:** no evidence supplied -- set `live_evidence` and re-run to judge it. "
                  "Until then: **unverified, not trusted.**")
else:
    lines.append(f"proposed role `{gen['role_label']}`")
    lines.append(f"> {gen['mechanism_description']}")
    judge = judge_hypothesis(live_material, gen["role_label"], gen["mechanism_description"],
                              live_evidence, "acss_bonus_judge")
    lines.append(f"\n**Judge (live GPT, `{judge.get('_source')}`):** confidence *{judge['confidence']}*")
    lines.append(f"> {judge['reasoning']}")
    lines.append(f"\n**Live outcome:** {LIVE_BADGE[judge['supported']]}")

display(Markdown("\n\n".join(lines)))
